# TP3 : embedding

In [ ]:
from gensim.models.phrases import Phrases, Phraser
from gensim.models import Word2Vec

#import nltk
from nltk.tokenize import wordpunct_tokenize
from unidecode import unidecode

In [ ]:
# chargement

class MySentences(object):
    """Tokenize and Lemmatize sentences"""
    def __init__(self, filename):
        self.filename = filename

    def __iter__(self):
        for line in open(self.filename, encoding='utf-8', errors="backslashreplace"):
            yield [unidecode(w.lower()) for w in wordpunct_tokenize(line)]

infile = f"../data/sents.txt"
sentences = MySentences(infile)

In [ ]:
bigram_phrases = Phrases(sentences)
type(bigram_phrases.vocab)
len(bigram_phrases.vocab.keys())

key_ = list(bigram_phrases.vocab.keys())[144]
print(key_)

bigram_phrases.vocab[key_]

bigram_phraser = Phraser(phrases_model=bigram_phrases)

In [ ]:
trigram_phrases = Phrases(bigram_phraser[sentences])
trigram_phraser = Phraser(phrases_model=trigram_phrases)

In [ ]:
corpus = list(trigram_phraser[bigram_phraser[sentences]])
print(corpus[:100])

# entrainement d'un modele word2vec sur ce corpus

In [ ]:
%%time
model = Word2Vec(
    corpus, # On passe le corpus de ngrams que nous venons de créer
    vector_size=32, # Le nombre de dimensions dans lesquelles le contexte des mots devra être réduit, aka. vector_size
    window=5, # La taille du "contexte", ici 5 mots avant et après le mot observé
    min_count=5, # On ignore les mots qui n'apparaissent pas au moins 5 fois dans le corpus
    workers=4, # Permet de paralléliser l'entraînement du modèle en 4 threads
    epochs=5 # Nombre d'itérations du réseau de neurones sur le jeu de données pour ajuster les paramètres avec la descente de gradient, aka. epochs.
)

In [ ]:
outfile = f"../data/newspapers.model"
model.save(outfile)

# explorer le modele

In [ ]:
# charger le modele en mémoire
model = Word2Vec.load("../data/newspapers.model")

# imprimer le vecteur d'un terme
model.wv["ministre"]

# calculer la similarité entre deux termes
model.wv.similarity("ministre", "roi")

# chercher les mots les plus proches d'un terme donné
model.wv.most_similar("ministre", topn=10)

# faire des recherches complexes à travers l'espace vectoriel
print(model.wv.most_similar(positive=['paris', 'londres','allemagne'], negative=['france', 'grande_bretagne'], topn=1))